In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

In [2]:
df = pd.read_csv("FlightPrice_train.csv")

In [3]:
df.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24-03-2019,Banglore,New Delhi,BLR ? DEL,22:20,22-03-2020 01:10,2h 50m,non-stop,No info,3897
1,Air India,01-05-2019,Kolkata,Banglore,CCU ? IXR ? BBI ? BLR,05:50,13:15,7h 25m,2 stops,No info,7662
2,Jet Airways,09-06-2019,Delhi,Cochin,DEL ? LKO ? BOM ? COK,09:25,10-06-2020 04:25,19h,2 stops,No info,13882
3,IndiGo,12-05-2019,Kolkata,Banglore,CCU ? NAG ? BLR,18:05,23:30,5h 25m,1 stop,No info,6218
4,IndiGo,01-03-2019,Banglore,New Delhi,BLR ? NAG ? DEL,16:50,21:35,4h 45m,1 stop,No info,13302


In [4]:
df.shape

(10683, 11)

# Data Preprocessing and Data Cleaning

## Convert Date

In [5]:
df["Journey_Day"] = pd.to_datetime(df["Date_of_Journey"], format="%d-%m-%Y").dt.day
df["Journey_Month"] = pd.to_datetime(df["Date_of_Journey"], format="%d-%m-%Y").dt.month

df.drop("Date_of_Journey", axis=1, inplace=True)

## Convert Departure Time

In [6]:
df["Dep_Hour"] = pd.to_datetime(df["Dep_Time"], format="%H:%M").dt.hour
df["Dep_Min"] = pd.to_datetime(df["Dep_Time"], format="%H:%M").dt.minute

## Convert Arrival Time

In [7]:
df["Arrival_Time"] = pd.to_datetime(df["Arrival_Time"], format='mixed', dayfirst=True)

df["Arrival_Hour"] = df["Arrival_Time"].dt.hour
df["Arrival_Min"] = df["Arrival_Time"].dt.minute

## Convert Duration to Minutes

In [10]:
def convert_duration(x):
    if pd.isnull(x):
        return 0
    
    h, m = 0, 0
    
    if 'h' in x:
        h = int(x.split('h')[0])
    if 'm' in x:
        m = int(x.split('h')[-1].replace('m','').strip())
        
    return h*60 + m

df["Duration"] = df["Duration"].apply(convert_duration)

## Convert Total Stops

In [11]:
df["Total_Stops"] = df["Total_Stops"].replace({
    "non-stop": 0,
    "1 stop": 1,
    "2 stops": 2,
    "3 stops": 3,
    "4 stops": 4
})

C:\Users\sj206\AppData\Local\Temp\ipykernel_10220\182790729.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Total_Stops"] = df["Total_Stops"].replace({


## Drop Unneccesary Columns

In [9]:
df.drop("Dep_Time", axis=1, inplace=True)

In [8]:
df.drop("Arrival_Time", axis=1, inplace=True)

In [12]:
df.drop(["Route", "Additional_Info"], axis=1, inplace=True)

# Feature Engineering

In [13]:
df["Journey_Date"] = pd.to_datetime(
    df["Journey_Day"].astype(str) + "-" + 
    df["Journey_Month"].astype(str) + "-2019",
    dayfirst=True
)

In [16]:
df["Is_Peak_Hour"] = df["Dep_Hour"].apply(lambda x: 1 if 9 <= x <= 21 else 0)

In [17]:
df.head()

,Airline,Source,Destination,Duration,Total_Stops,Price,Journey_Day,Journey_Month,Dep_Hour,Dep_Min,Arrival_Hour,Arrival_Min,Journey_Date,Is_Peak_Hour
0,IndiGo,Banglore,New Delhi,170,0,3897,24,3,22,20,1,10,2019-03-24,0
1,Air India,Kolkata,Banglore,445,2,7662,1,5,5,50,13,15,2019-05-01,0
2,Jet Airways,Delhi,Cochin,1140,2,13882,9,6,9,25,4,25,2019-06-09,1
3,IndiGo,Kolkata,Banglore,325,1,6218,12,5,18,5,23,30,2019-05-12,1
4,IndiGo,Banglore,New Delhi,285,1,13302,1,3,16,50,21,35,2019-03-01,1


In [18]:
print(df.shape)
print(df.info())
print(df.isnull().sum())

(10683, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Airline        10683 non-null  object        
 1   Source         10683 non-null  object        
 2   Destination    10683 non-null  object        
 3   Duration       10683 non-null  int64         
 4   Total_Stops    10683 non-null  int64         
 5   Price          10683 non-null  int64         
 6   Journey_Day    10683 non-null  int32         
 7   Journey_Month  10683 non-null  int32         
 8   Dep_Hour       10683 non-null  int32         
 9   Dep_Min        10683 non-null  int32         
 10  Arrival_Hour   10683 non-null  int32         
 11  Arrival_Min    10683 non-null  int32         
 12  Journey_Date   10683 non-null  datetime64[ns]
 13  Is_Peak_Hour   10683 non-null  int64         
dtypes: datetime64[ns](1), int32(6), int64(4), object(3)
memory

In [21]:
df.to_csv("Cleaned_FlightPrice_train.csv", index=False)